# Notebook 4 v2 — Windows and Features for the 9-variant scenario matrix

**What changed from the NB4 A/F notebook:** the input is now NB3 v2 (9 variants per episode: warming and cooling events, ramp and step drift). Windows are built per (episode, **variant**) instead of per (episode, scenario). Feature definitions, window sizes, reference statistics and validation are **unchanged**, and the four variants that already existed in v1 must reproduce the v1 feature table exactly (section 5).

**What this notebook does:** turns the Notebook 3 v2 scenarios into 30-min causal windows and computes **19 features** per window:
the 16 original features (`peer_agreement` renamed `level_closeness`, definition unchanged) plus 3 new ones: `agree_sign`, `dev_mean_adj`, `peer_level_rel`.

- **Set A** = the 16 original features. **Set F** = 15 features: A minus the 4 absolute-level features (`t_mean`, `t_min`, `t_max`, `peer_median_level`) plus the 3 new ones.
- Both sets come from the **same windows and labels**, so they can be compared fairly in Notebook 5.
- The 3 new features use reference statistics (`sigma_j`, per-target offset, time-of-day baseline) fitted on **real, training-period data only**.

**Inputs (read-only):** `scenario_dataset_v2.parquet`, `scenario_metadata_v2.csv`, `scenario_config_v2.json` (NB3 v2), the cleaned real data from NB2 (`REAL_FILE`), and, for comparison only, the v1 outputs `feature_dataset_AF.parquet` and `reference_stats.json`.
**Outputs (new files):** `feature_dataset_AF_v2.parquet`, `feature_config_AF_v2.json`, `reference_stats_v2.json`. The v1 files are never overwritten.

**Not here:** no classifier, no tuning, no accuracy numbers.

**Debugging tips:** every section prints `PASS`/`OK`/`WARN` lines and stops with `CHECK FAILED: ...` at the first real problem. Sections are independent small cells; the window arrays are kept in `ARR`, the final table in `T`.

In [2]:
from pathlib import Path
import json, hashlib, warnings
import numpy as np
import pandas as pd

try:
    import pyarrow  # noqa: F401
except ImportError:
    raise ImportError("pyarrow is missing. In your Jupyter environment run: conda install -c conda-forge pyarrow  (then restart the kernel)")

# Project root = first parent folder that contains data/processed
_here = Path.cwd().resolve()
ROOT = next((p for p in [_here, *_here.parents] if (p / "data" / "processed").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find data/processed above " + str(_here) + ". Start Jupyter inside the project folder.")
PROC = ROOT / "data" / "processed"
OUT = ROOT / "outputs" / "notebook04"
OUT.mkdir(parents=True, exist_ok=True)

def sha256(path, chunk=1 << 24):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()

def check(cond, msg):
    if not bool(cond):
        raise ValueError("CHECK FAILED: " + msg)
    print("PASS ", msg)

def warn(cond, msg):
    print("OK   " if cond else "WARN ", msg)

def same(a, b, tol=1e-9):
    a, b = float(a), float(b)
    return (np.isnan(a) and np.isnan(b)) or abs(a - b) <= tol

print("ROOT:", ROOT)

ROOT: C:\AIoT_Research


In [3]:
CFG = {
    "value_column": "modified_value",   # observed signal (never original_value or the injected components)
    "short_window": 30, "long_window": 60, "stride": 5,   # minutes
    "min_target_obs_short": 10, "min_target_obs_long": 20,
    "min_pairs_short": 10, "min_pairs_long": 20,
    "min_peers_per_minute": 2,           # peer reference exists only if >= 2 of the peers are valid
    "purity_cutoff": 1.0,                # "pure" window = whole short window has one label
    "agreement_threshold_c": 1.0,        # level_closeness: |target - peer| <= 1.0 C  (old peer_agreement)
    "kappa": 1.0, "change_lag": 5,       # agree_sign: 5-minute changes, peer change must exceed kappa * sigma_j
}
assert CFG["long_window"] >= CFG["short_window"] + CFG["change_lag"]   # agree_sign needs W + lag minutes of history
W, L, STRIDE = CFG["short_window"], CFG["long_window"], CFG["stride"]
LABELS = ["NORMAL", "DRIFT", "EVENT", "EVENT_DRIFT"]

# --- Real (NB2) cleaned data: used ONLY for train-period reference statistics. Edit here if your names differ.
REAL_FILE = PROC / "clean_sensor_data.parquet"
REAL_TEMP_COL = "temperature_clean"
TRAIN_END_FALLBACK = "2004-03-15 02:41"      # used only if the real file has no 'split' column
V1_FEATURE_FILE = PROC / "feature_dataset_AF.parquet"      # v1 NB4 A/F output (for the identity check)
V1_REF_FILE = PROC / "reference_stats.json"                # v1 reference statistics (must be identical: real data did not change)
V1_VARIANTS = {"NORMAL": "NORMAL", "DRIFT_RAMP": "DRIFT", "EVENT_WARM": "EVENT", "ED_WARM_RAMP": "EVENT_DRIFT"}   # v2 variant -> v1 scenario name
VARIANT_ORDER = ["NORMAL", "DRIFT_RAMP", "DRIFT_STEP", "EVENT_WARM", "EVENT_COOL", "ED_WARM_RAMP", "ED_WARM_STEP", "ED_COOL_RAMP", "ED_COOL_STEP"]

A_FEATURES = ["t_mean", "t_std", "t_min", "t_max", "t_mean_diff", "t_mean_abs_diff", "t_slope", "t_long_slope",
              "peer_median_level", "peer_slope", "dev_mean", "dev_std", "dev_slope", "dev_long_slope",
              "level_closeness", "peer_corr"]
NEW_FEATURES = ["agree_sign", "dev_mean_adj", "peer_level_rel"]
ABS_LEVEL = ["t_mean", "t_min", "t_max", "peer_median_level"]
F_FEATURES = [f for f in A_FEATURES if f not in ABS_LEVEL] + NEW_FEATURES
ALL_FEATURES = A_FEATURES + NEW_FEATURES
assert len(A_FEATURES) == 16 and len(F_FEATURES) == 15 and len(ALL_FEATURES) == 19
print("A:", len(A_FEATURES), " F:", len(F_FEATURES), " all:", len(ALL_FEATURES))

A: 16  F: 15  all: 19


## 1. Load inputs (read-only)

In [5]:
NB3_FILES = [PROC / "scenario_dataset_v2.parquet", PROC / "scenario_metadata_v2.csv", PROC / "scenario_config_v2.json"]
for p in NB3_FILES:
    check(p.exists() and p.stat().st_size > 0, f"input exists: {p.name}")
READ_ONLY = NB3_FILES + [p for p in (V1_FEATURE_FILE, V1_REF_FILE) if p.exists()]
HASH_BEFORE = {p.name: sha256(p) for p in READ_ONLY}

S = pd.read_parquet(PROC / "scenario_dataset_v2.parquet")
M = pd.read_csv(PROC / "scenario_metadata_v2.csv", parse_dates=["context_start", "scenario_start", "scenario_end"])
with open(PROC / "scenario_config_v2.json") as f:
    NB3 = json.load(f)
need = ["episode_id", "scenario", "variant", "interval_phase", "event_direction", "drift_type", "label", "split", "moteid", "timestamp", "modified_value", "event_component", "drift_component"]
check(set(need) <= set(S.columns), "scenario_dataset has required columns")
check(set(S["scenario"]) == set(LABELS), "scenario (class) names valid")
check(set(S["variant"]) == set(VARIANT_ORDER) and S.groupby("episode_id")["variant"].nunique().eq(9).all(), "all 9 variants present in every episode")
PEERS_BY_TARGET = NB3["candidate_peers"]      # {"target_mote": [peer motes]}
print("scenario rows:", len(S), "| episodes:", S["episode_id"].nunique(), "| variants:", S["variant"].nunique(), "| peer method:", NB3["peer_method"])

# Real NB2 data
if not REAL_FILE.exists():
    raise FileNotFoundError(f"{REAL_FILE.name} not found. Parquet files in data/processed: {[p.name for p in PROC.glob('*.parquet')]}. Edit REAL_FILE in the config cell.")
REAL = pd.read_parquet(REAL_FILE)
miss = {"moteid", "timestamp", REAL_TEMP_COL} - set(REAL.columns)
if miss:
    raise KeyError(f"{REAL_FILE.name} is missing columns {miss}. Columns found: {list(REAL.columns)}. Edit REAL_TEMP_COL in the config cell.")
REAL["timestamp"] = pd.to_datetime(REAL["timestamp"])
if "split" not in REAL.columns:
    print("WARN: no 'split' column in real file; using TRAIN_END_FALLBACK =", TRAIN_END_FALLBACK)
    REAL["split"] = np.where(REAL["timestamp"] <= pd.Timestamp(TRAIN_END_FALLBACK), "train", "other")
REAL = REAL[["moteid", "timestamp", "split", REAL_TEMP_COL]].rename(columns={REAL_TEMP_COL: "temp"})
REAL["moteid"] = REAL["moteid"].astype("int64")       # NB2 stores int16 / float32; compute in 64-bit so all checks agree exactly
REAL["temp"] = REAL["temp"].astype("float64")
print("real rows:", len(REAL), "| by split:", REAL["split"].value_counts().to_dict())

PASS  input exists: scenario_dataset_v2.parquet
PASS  input exists: scenario_metadata_v2.csv
PASS  input exists: scenario_config_v2.json
PASS  scenario_dataset has required columns
PASS  scenario (class) names valid
PASS  all 9 variants present in every episode
scenario rows: 341010 | episodes: 18 | variants: 9 | peer method: train-period temperature correlation (no location file available)
real rows: 1785348 | by split: {'train': 1249776, 'validation': 267786, 'test': 267786}


## 2. Reference statistics (fitted on real TRAIN data only)
- `sigma_j` = std of mote *j*'s 5-minute temperature changes (train).
- **Offset** per target = median over train days of the daily median of (target − peer median). Used for `dev_mean_adj = dev_mean − offset`.
- **Baseline** per target and hour = median over train days of the hourly mean peer reference. Used for `peer_level_rel = peer_median_level − baseline(hour of window end)`.
- For **train** windows the window's own day(s) are left out (so training windows are not favoured). Validation/test windows use all train days.

In [7]:
def peer_ref_df(wide, peers, cfg):
    # median of the valid peers per minute; NaN if fewer than min_peers_per_minute are valid
    P = wide.reindex(columns=[int(p) for p in peers])
    return P.median(axis=1).where(P.notna().sum(axis=1) >= cfg["min_peers_per_minute"])

def fit_reference(real, peers_by_target, cfg):
    tr = real[real["split"] == "train"]                      # <- the ONLY place real data is filtered by split
    wide = tr.pivot(index="timestamp", columns="moteid", values="temp").sort_index().asfreq("min")
    sigma = (wide - wide.shift(cfg["change_lag"])).std(axis=0, ddof=0)
    dev_daily, hour_table = {}, {}
    for t_key, peers in peers_by_target.items():
        t = int(t_key)
        ref = peer_ref_df(wide, peers, cfg)
        dev = (wide[t] - ref).dropna() if t in wide.columns else pd.Series(dtype=float, index=pd.DatetimeIndex([]))
        dev_daily[t] = dev.groupby(dev.index.floor("D")).median()
        rv = ref.dropna()
        hour_table[t] = rv.groupby([rv.index.floor("D"), rv.index.hour]).mean().unstack() if len(rv) else pd.DataFrame()
    return {"sigma": {int(k): float(v) for k, v in sigma.items()}, "dev_daily": dev_daily, "hour_table": hour_table}

def lookup(ref, target, start_ts, end_ts, split):
    # offset and baseline for one window; train windows leave out their own day(s)
    drop = [start_ts.floor("D"), end_ts.floor("D")] if split == "train" else []
    dev = ref["dev_daily"][target].drop(index=drop, errors="ignore")
    offset = float(dev.median()) if len(dev) else np.nan
    tab, baseline = ref["hour_table"][target], np.nan
    if end_ts.hour in tab.columns:
        col = tab[end_ts.hour].drop(index=drop, errors="ignore").dropna()
        baseline = float(col.median()) if len(col) else np.nan
    return offset, baseline

REF = fit_reference(REAL, PEERS_BY_TARGET, CFG)
sig = pd.Series(REF["sigma"])
print("sigma_j: motes =", len(sig), "| NaN (no train data):", sig[sig.isna()].index.tolist(), "| median %.3f C, range %.3f to %.3f" % (sig.median(), sig.min(), sig.max()))
no_dev = [t for t, s in REF["dev_daily"].items() if len(s) == 0]
print("targets without train offset:", no_dev, "| days used for offsets (median per target):", int(np.median([len(s) for s in REF["dev_daily"].values()])))

sigma_j: motes = 54 | NaN (no train data): [5] | median 0.215 C, range 0.136 to 0.416
targets without train offset: [] | days used for offsets (median per target): 17


## 3. Window features
One function, `window_features(y, P, end, ...)`. It slices only `[end-L+1 : end+1]`, so it cannot see the future (tested in section 6). Missing data stays NaN and the reason is recorded; nothing is interpolated.

`agree_sign` (paper Eq. 2): for each minute *m* in the 30-min window, `change(m) = x(m) − x(m−5)` for the target and for each peer. A (target, peer, minute) pair counts only if both changes exist; it *agrees* if the signs are equal and nonzero and `|peer change| > kappa × sigma_j`. `agree_sign` = agreeing pairs / valid pairs (NaN if fewer than 10 pairs).

In [9]:
def ols_slope(v, min_obs):
    m = np.isfinite(v)
    if m.sum() < min_obs:
        return np.nan, "too_few_obs"
    t = np.arange(len(v), dtype=float)[m]; yv = v[m]; tc = t - t.mean()
    den = float((tc ** 2).sum())
    return (float((tc * (yv - yv.mean())).sum() / den), "") if den > 0 else (np.nan, "degenerate_time")

def peer_reference(P, min_peers):
    ok = np.isfinite(P); cnt = ok.sum(axis=1)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        med = np.nanmedian(np.where(ok, P, np.nan), axis=1)
    return np.where(cnt >= min_peers, med, np.nan), cnt

def window_features(y, P, end, cfg, sig_vec, offset, baseline):
    # y: target (n,), P: peers (n,k), end: window-end index, sig_vec: sigma_j per peer (k,), offset/baseline: train-derived scalars
    w, l = cfg["short_window"], cfg["long_window"]
    ys, Ps = y[end - w + 1: end + 1], P[end - w + 1: end + 1]
    yl, Pl = y[end - l + 1: end + 1], P[end - l + 1: end + 1]
    f = {k: np.nan for k in ALL_FEATURES}; why = {}

    def put_slope(name, v, min_obs):
        f[name], r = ols_slope(v, min_obs)
        if r: why[name] = r

    # target features
    valid = ys[np.isfinite(ys)]; n = len(valid)
    if n >= cfg["min_target_obs_short"]:
        d = np.diff(valid)
        f.update(t_mean=valid.mean(), t_std=valid.std(), t_min=valid.min(), t_max=valid.max(),
                 t_mean_diff=d.mean(), t_mean_abs_diff=np.abs(d).mean())
    else:
        for k in ["t_mean", "t_std", "t_min", "t_max", "t_mean_diff", "t_mean_abs_diff"]:
            why[k] = "too_few_target_obs"
    put_slope("t_slope", ys, cfg["min_target_obs_short"])
    put_slope("t_long_slope", yl, cfg["min_target_obs_long"])

    # peer reference and deviation
    ref_s, cnt_s = peer_reference(Ps, cfg["min_peers_per_minute"])
    ref_l, _ = peer_reference(Pl, cfg["min_peers_per_minute"])
    dev_s, dev_l = ys - ref_s, yl - ref_l
    if np.isfinite(ref_s).sum() >= cfg["min_pairs_short"]:
        f["peer_median_level"] = float(np.nanmean(ref_s))
    else:
        why["peer_median_level"] = "too_few_peer_obs"
    put_slope("peer_slope", ref_s, cfg["min_pairs_short"])
    npairs = int(np.isfinite(dev_s).sum())
    if npairs >= cfg["min_pairs_short"]:
        f["dev_mean"], f["dev_std"] = float(np.nanmean(dev_s)), float(np.nanstd(dev_s))
    else:
        why["dev_mean"] = why["dev_std"] = "too_few_pairs"
    put_slope("dev_slope", dev_s, cfg["min_pairs_short"])
    put_slope("dev_long_slope", dev_l, cfg["min_pairs_long"])

    # level_closeness (old peer_agreement): share of valid (target, peer, minute) pairs within the threshold
    pair_ok = np.isfinite(Ps) & np.isfinite(ys)[:, None]
    if pair_ok.sum() >= cfg["min_pairs_short"]:
        close = np.abs(ys[:, None] - Ps) <= cfg["agreement_threshold_c"]
        f["level_closeness"] = float((close & pair_ok).sum() / pair_ok.sum())
    else:
        why["level_closeness"] = "too_few_pairs"

    # peer_corr
    m = np.isfinite(ys) & np.isfinite(ref_s)
    if m.sum() < cfg["min_pairs_short"]:
        why["peer_corr"] = "too_few_pairs"
    elif ys[m].std() == 0 or ref_s[m].std() == 0:
        why["peer_corr"] = "constant_signal"
    else:
        f["peer_corr"] = float(np.corrcoef(ys[m], ref_s[m])[0, 1])

    # NEW: agree_sign (paper Eq. 2) on 5-minute changes, past data only
    idx = np.arange(end - w + 1, end + 1)
    dy = y[idx] - y[idx - cfg["change_lag"]]
    dp = P[idx] - P[idx - cfg["change_lag"]]
    ok = np.isfinite(dy)[:, None] & np.isfinite(dp) & np.isfinite(sig_vec)[None, :]
    if ok.sum() >= cfg["min_pairs_short"]:
        match = (dy[:, None] * dp > 0) & (np.abs(dp) > cfg["kappa"] * sig_vec[None, :])
        f["agree_sign"] = float((match & ok).sum() / ok.sum())
    else:
        why["agree_sign"] = "too_few_change_pairs"

    # NEW: offset-corrected deviation and baseline-relative peer level
    if np.isfinite(f["dev_mean"]) and np.isfinite(offset):
        f["dev_mean_adj"] = float(f["dev_mean"] - offset)
    else:
        why["dev_mean_adj"] = "missing_dev_mean_or_offset"
    if np.isfinite(f["peer_median_level"]) and np.isfinite(baseline):
        f["peer_level_rel"] = float(f["peer_median_level"] - baseline)
    else:
        why["peer_level_rel"] = "missing_peer_level_or_baseline"

    q = {"n_target_valid": n, "target_missing_fraction": 1 - n / w, "n_target_valid_long": int(np.isfinite(yl).sum()),
         "mean_valid_peers": float(cnt_s.mean()), "n_dev_pairs": npairs}
    return f, q, why
print("window_features defined")

window_features defined


## 4. Build the windows (one pass over every episode × variant)

In [11]:
ARR = {}      # (episode, variant) -> arrays, kept for the checks below
rows = []
meta = M.set_index(["episode_id", "variant"])
for (ep, scn), g in S.groupby(["episode_id", "variant"], sort=True):      # scn = variant name here
    mrow = meta.loc[(ep, scn)]
    target = int(mrow["target_moteid"])
    peers = [int(x) for x in str(mrow["peer_moteids"]).split(",")]
    assert peers == PEERS_BY_TARGET[str(target)] and target not in peers, f"peer mismatch in {ep}"
    wide = g.pivot(index="timestamp", columns="moteid", values=CFG["value_column"]).sort_index()
    ts = wide.index
    assert (np.diff(ts.values).astype("timedelta64[m]").astype(int) == 1).all(), "minute grid not contiguous"
    y, P = wide[target].to_numpy(float), wide[peers].to_numpy(float)
    sig_vec = np.array([REF["sigma"].get(p, np.nan) for p in peers], dtype=float)
    tg = g[g["moteid"] == target].set_index("timestamp").reindex(ts)
    lab = tg["label"].to_numpy()
    ARR[(ep, scn)] = {"y": y, "P": P, "ts": ts, "peers": peers, "target": target, "sig": sig_vec}
    for end in range(L - 1, len(ts), STRIDE):
        offset, baseline = lookup(REF, target, ts[end - W + 1], ts[end], mrow["split"])
        f, q, why = window_features(y, P, end, CFG, sig_vec, offset, baseline)
        purity = float((lab[end - W + 1: end + 1] == lab[end]).mean())
        long_purity = float((lab[end - L + 1: end + 1] == lab[end]).mean())
        rows.append({
            "window_id": f"{ep}_{scn}_{ts[end]:%Y%m%d%H%M}", "episode_id": ep, "split": mrow["split"], "scenario": mrow["scenario"], "variant": scn,
            "event_direction": mrow["event_direction"], "drift_type": mrow["drift_type"], "drift_event_relation": mrow["drift_event_relation"],
            "interval_phase": tg["interval_phase"].iloc[end],
            "label": lab[end], "label_purity": purity, "long_label_purity": long_purity,
            "window_group": "pure" if purity >= CFG["purity_cutoff"] else "transition",
            "window_pure": purity >= CFG["purity_cutoff"], "long_window_pure": long_purity == 1.0,
            "minutes_since_onset": (ts[end] - mrow["scenario_start"]).total_seconds() / 60,
            "moteid": target, "peer_moteids": ",".join(map(str, peers)),
            "window_start": ts[end - W + 1], "window_end": ts[end], "long_window_start": ts[end - L + 1],
            "end_index": end, "offset_used": offset, "baseline_used": baseline,      # kept for debugging/checks, NOT features
            **f, **q,
            "nan_reasons": ";".join(f"{k}:{v}" for k, v in sorted(why.items())),
            "gt_event_component": float(tg["event_component"].iloc[end]),             # ground truth for audit only
            "gt_drift_component": float(tg["drift_component"].iloc[end]),
        })
T = pd.DataFrame(rows)
T["F_complete"] = T[F_FEATURES].notna().all(axis=1)
T["A_complete"] = T[A_FEATURES].notna().all(axis=1)
print("windows:", len(T), "| by split:", T["split"].value_counts().to_dict())
print("labels :", T["label"].value_counts().reindex(LABELS).to_dict())
print("pure   :", int(T["window_pure"].sum()), " transition:", int((~T["window_pure"]).sum()))
warn(len(T) == 162 * 73, "window count is 11,826 = 162 scenarios x 73 windows (v1 had 72 x 73 = 5,256)")

windows: 11826 | by split: {'train': 6570, 'validation': 2628, 'test': 2628}
labels : {'NORMAL': 1458, 'DRIFT': 2592, 'EVENT': 2592, 'EVENT_DRIFT': 5184}
pure   : 11106  transition: 720
OK    window count is 11,826 = 162 scenarios x 73 windows (v1 had 72 x 73 = 5,256)


## 5. Checks A — v1 variants reproduce v1, and train-only reference statistics

In [13]:
# 5.1 The four variants that existed in v1 must reproduce the v1 feature table EXACTLY (windows, labels and all 19 features)
if V1_FEATURE_FILE.exists():
    V1 = pd.read_parquet(V1_FEATURE_FILE)
    sub = T[T["variant"].isin(V1_VARIANTS)].copy()
    sub["v1_id"] = sub["episode_id"] + "_" + sub["variant"].map(V1_VARIANTS) + "_" + sub["window_end"].dt.strftime("%Y%m%d%H%M")
    check(len(sub) == len(V1) and set(sub["v1_id"]) == set(V1["window_id"]), f"same {len(V1)} windows as the v1 feature table for the 4 shared variants")
    J = sub.set_index("v1_id").loc[V1["window_id"]]
    check((J["label"].values == V1["label"].values).all() and (J["split"].values == V1["split"].values).all() and (J["window_pure"].values == V1["window_pure"].values).all(), "same labels, splits and pure/transition flags as v1")
    # dev_mean_adj and peer_level_rel use reference statistics fitted on the real NB2 data. NB2 stores float32; the v1 table may have been
    # computed in float32 and this notebook computes in float64, so those two may differ by float32 rounding (~1e-6). All other features must match to 1e-9.
    REF_BASED = ("dev_mean_adj", "peer_level_rel")
    for c in ALL_FEATURES:
        tol = 1e-5 if c in REF_BASED else 1e-9
        x, y = J[c].to_numpy(float), V1[c].to_numpy(float)
        check(np.allclose(x, y, rtol=0, atol=tol, equal_nan=True), f"feature identical to v1 within {tol:g}: {c}" + (f" (max |diff| {np.nanmax(np.abs(x - y)):.1e}, float32 rounding)" if c in REF_BASED else ""))
    check((J["nan_reasons"].values == V1["nan_reasons"].values).all(), "same missing-value reasons as v1")
else:
    print("WARN v1 feature_dataset_AF.parquet not found -> identity check SKIPPED")

# 5.2 Reference statistics use train data only
tr_max = REAL.loc[REAL["split"] == "train", "timestamp"].max()
other_min = REAL.loc[REAL["split"] != "train", "timestamp"].min()
check(pd.isna(other_min) or tr_max < other_min, f"train data ends ({tr_max}) before validation/test data begins ({other_min})")
latest_day = max(s.index.max() for s in REF["dev_daily"].values() if len(s))
check(latest_day <= tr_max, "latest day used for offsets is inside the train period")

# 5.3 Replace ALL non-train real data by garbage: the statistics must not change
REAL_bad = REAL.copy()
REAL_bad.loc[REAL_bad["split"] != "train", "temp"] = 999.0
REF_bad = fit_reference(REAL_bad, PEERS_BY_TARGET, CFG)
check(np.allclose(sig.to_numpy(float), pd.Series(REF_bad["sigma"]).reindex(sig.index).to_numpy(float), equal_nan=True), "sigma_j unchanged when validation/test real data is garbage")
check(all(REF["dev_daily"][t].equals(REF_bad["dev_daily"][t]) for t in REF["dev_daily"]), "daily offsets unchanged when validation/test real data is garbage")
check(all(REF["hour_table"][t].equals(REF_bad["hour_table"][t]) for t in REF["hour_table"]), "hourly baselines unchanged when validation/test real data is garbage")

PASS  same 5256 windows as the v1 feature table for the 4 shared variants
PASS  same labels, splits and pure/transition flags as v1
PASS  feature identical to v1 within 1e-09: t_mean
PASS  feature identical to v1 within 1e-09: t_std
PASS  feature identical to v1 within 1e-09: t_min
PASS  feature identical to v1 within 1e-09: t_max
PASS  feature identical to v1 within 1e-09: t_mean_diff
PASS  feature identical to v1 within 1e-09: t_mean_abs_diff
PASS  feature identical to v1 within 1e-09: t_slope
PASS  feature identical to v1 within 1e-09: t_long_slope
PASS  feature identical to v1 within 1e-09: peer_median_level
PASS  feature identical to v1 within 1e-09: peer_slope
PASS  feature identical to v1 within 1e-09: dev_mean
PASS  feature identical to v1 within 1e-09: dev_std
PASS  feature identical to v1 within 1e-09: dev_slope
PASS  feature identical to v1 within 1e-09: dev_long_slope
PASS  feature identical to v1 within 1e-09: level_closeness
PASS  feature identical to v1 within 1e-09: pee

## 6. Checks B — causality and independent recomputation

In [15]:
# 6.1 Causality: overwrite everything AFTER the window end with garbage; ALL 19 features must not change
sample = T.iloc[::13]
n_checked = 0
for r in sample.itertuples():
    a = ARR[(r.episode_id, r.variant)]
    y2, P2 = a["y"].copy(), a["P"].copy()
    y2[r.end_index + 1:] = 1e6; P2[r.end_index + 1:] = -1e6
    f2, _, _ = window_features(y2, P2, r.end_index, CFG, a["sig"], r.offset_used, r.baseline_used)
    for k in ALL_FEATURES:
        assert same(f2[k], getattr(r, k)), f"feature {k} changed when future data was corrupted ({r.window_id})"
    n_checked += 1
check(n_checked > 0, f"causality: {n_checked} windows x 19 features unchanged under future-data corruption (also re-confirms the stored values)")

PASS  causality: 910 windows x 19 features unchanged under future-data corruption (also re-confirms the stored values)


In [16]:
# 6.2 Independent recomputation with pandas (different code path from window_features) for 40 random windows
def recompute(r):
    a = ARR[(r.episode_id, r.variant)]; peers = a["peers"]; e = r.end_index
    df = pd.DataFrame(a["P"], columns=peers); df["y"] = a["y"]
    win = df.iloc[e - W + 1: e + 1]; Pw = win[peers]
    out = {}
    if win["y"].notna().sum() >= CFG["min_target_obs_short"]:
        out["t_mean"] = win["y"].mean()
    ref = Pw.median(axis=1).where(Pw.notna().sum(axis=1) >= CFG["min_peers_per_minute"])
    dev = win["y"] - ref
    if ref.notna().sum() >= CFG["min_pairs_short"]:
        out["peer_median_level"] = ref.mean()
    if dev.notna().sum() >= CFG["min_pairs_short"]:
        out["dev_mean"] = dev.mean()
    ok = Pw.notna().values & win["y"].notna().values[:, None]
    if ok.sum() >= CFG["min_pairs_short"]:
        close = np.abs(win["y"].values[:, None] - Pw.values) <= CFG["agreement_threshold_c"]
        out["level_closeness"] = (close & ok).sum() / ok.sum()
    full = df.iloc[e - W - CFG["change_lag"] + 1: e + 1]
    dy = full["y"].diff(CFG["change_lag"]).iloc[-W:]; dp = full[peers].diff(CFG["change_lag"]).iloc[-W:]
    s = a["sig"]
    ok2 = dy.notna().values[:, None] & dp.notna().values & np.isfinite(s)[None, :]
    if ok2.sum() >= CFG["min_pairs_short"]:
        sy = np.sign(dy.values)[:, None]
        match = (sy != 0) & (sy == np.sign(dp.values)) & (dp.abs().values > CFG["kappa"] * s[None, :])
        out["agree_sign"] = (match & ok2).sum() / ok2.sum()
    if "dev_mean" in out and np.isfinite(r.offset_used):
        out["dev_mean_adj"] = out["dev_mean"] - r.offset_used
    if "peer_median_level" in out and np.isfinite(r.baseline_used):
        out["peer_level_rel"] = out["peer_median_level"] - r.baseline_used
    return out

CHECK_KEYS = ["t_mean", "peer_median_level", "dev_mean", "level_closeness", "agree_sign", "dev_mean_adj", "peer_level_rel"]
n_vals = 0
for r in T.sample(40, random_state=2026).itertuples():
    exp = recompute(r)
    for k in CHECK_KEYS:
        assert same(exp.get(k, np.nan), getattr(r, k)), f"{k} differs in {r.window_id}: independent={exp.get(k, np.nan)} stored={getattr(r, k)}"
        n_vals += 1
check(True, f"independent recomputation: {n_vals} values match (7 features x 40 windows)")

PASS  independent recomputation: 280 values match (7 features x 40 windows)


In [17]:
# 6.3 Independent recomputation of offset and baseline straight from the real train data (resample-based)
tr = REAL[REAL["split"] == "train"]
wide_tr = tr.pivot(index="timestamp", columns="moteid", values="temp").sort_index().asfreq("min")
n_ref = 0
for r in pd.concat([T[T["split"] == "train"].sample(8, random_state=1), T[T["split"] != "train"].sample(8, random_state=1)]).itertuples():
    ref = peer_ref_df(wide_tr, PEERS_BY_TARGET[str(r.moteid)], CFG)
    dev = (wide_tr[r.moteid] - ref).dropna() if r.moteid in wide_tr.columns else pd.Series(dtype=float)
    days = [pd.Timestamp(r.window_start).floor("D"), pd.Timestamp(r.window_end).floor("D")] if r.split == "train" else []
    daily = dev.resample("D").median().dropna() if len(dev) else dev
    off = float(daily.drop(index=days, errors="ignore").median()) if len(daily.drop(index=days, errors="ignore")) else np.nan
    hourly = ref.resample("h").mean().dropna()
    hourly = hourly[~hourly.index.floor("D").isin(days)]
    hh = hourly[hourly.index.hour == pd.Timestamp(r.window_end).hour]
    base = float(hh.median()) if len(hh) else np.nan
    assert same(off, r.offset_used), f"offset differs in {r.window_id}: independent={off} stored={r.offset_used}"
    assert same(base, r.baseline_used), f"baseline differs in {r.window_id}: independent={base} stored={r.baseline_used}"
    n_ref += 1
check(True, f"independent offset/baseline recomputation matches for {n_ref} windows (train windows with leave-day-out, validation/test with all train days)")

PASS  independent offset/baseline recomputation matches for 16 windows (train windows with leave-day-out, validation/test with all train days)


## 7. Checks C — structure, leakage, missing data

In [19]:
FORBIDDEN = {"variant", "event_direction", "drift_type", "drift_event_relation", "interval_phase", "label", "scenario", "episode_id", "split", "label_purity", "long_label_purity", "window_pure", "long_window_pure",
             "window_group", "minutes_since_onset", "gt_event_component", "gt_drift_component", "offset_used", "baseline_used", "end_index"}
check(not (FORBIDDEN & set(ALL_FEATURES)), "no label/ground-truth/metadata column is used as a feature")
check(set(F_FEATURES) <= set(A_FEATURES + NEW_FEATURES) and set(ABS_LEVEL).isdisjoint(F_FEATURES), "F excludes the 4 absolute-level features")
check(np.isfinite(T[ALL_FEATURES].to_numpy(float)[~np.isnan(T[ALL_FEATURES].to_numpy(float))]).all(), "no infinite feature values")
check(T["window_id"].is_unique, "window ids are unique")
check(T.loc[T[ALL_FEATURES].isna().any(axis=1), "nan_reasons"].ne("").all(), "every window with a missing feature has a recorded reason")
for c in ["level_closeness", "agree_sign"]:
    check(T[c].dropna().between(0, 1).all(), f"{c} lies in [0, 1]")
check(T["label"].isin(LABELS).all() and T["label_purity"].between(0, 1).all(), "labels valid, purity in [0, 1]")
bounds = S.groupby("split")["timestamp"].agg(["min", "max"])
check(all((T.loc[T["split"] == s, "long_window_start"].min() >= bounds.loc[s, "min"]) and (T.loc[T["split"] == s, "window_end"].max() <= bounds.loc[s, "max"]) for s in bounds.index),
      "every window lies inside its own split (no window crosses splits)")
check(set(T.loc[T["split"] == "train", "moteid"]).isdisjoint(set(T.loc[T["split"] != "train", "moteid"])), "target motes of train windows never appear as validation/test targets")
check(T.groupby("episode_id")["variant"].nunique().eq(9).all(), "every episode has windows for all 9 variants")
check(T.groupby(["episode_id", "variant"]).size().groupby("episode_id").nunique().eq(1).all(), "every variant of an episode has the same number of windows")
check(T.groupby("variant")["scenario"].nunique().eq(1).all() and set(T["scenario"]) == set(LABELS), "each variant maps to exactly one class")
print("\nMissing values (% of all windows):")
miss = (100 * T[ALL_FEATURES].isna().mean()).round(2).sort_values(ascending=False)
print(miss[miss > 0].to_string() if (miss > 0).any() else "none")
print("\nWindows with all A features:", int(T["A_complete"].sum()), "| with all F features:", int(T["F_complete"].sum()), "of", len(T))
print("\nMost common NaN reasons:\n" + T.loc[T["nan_reasons"] != "", "nan_reasons"].value_counts().head(8).to_string())

PASS  no label/ground-truth/metadata column is used as a feature
PASS  F excludes the 4 absolute-level features
PASS  no infinite feature values
PASS  window ids are unique
PASS  every window with a missing feature has a recorded reason
PASS  level_closeness lies in [0, 1]
PASS  agree_sign lies in [0, 1]
PASS  labels valid, purity in [0, 1]
PASS  every window lies inside its own split (no window crosses splits)
PASS  target motes of train windows never appear as validation/test targets
PASS  every episode has windows for all 9 variants
PASS  every variant of an episode has the same number of windows
PASS  each variant maps to exactly one class

Missing values (% of all windows):
agree_sign           3.42
peer_corr            3.12
dev_mean_adj         3.12
dev_mean             3.12
dev_std              3.12
dev_slope            3.12
t_mean               2.36
t_min                2.36
t_max                2.36
t_mean_diff          2.36
t_mean_abs_diff      2.36
t_slope              2.36


In [20]:
# Descriptive look at the 3 new features by label (pure windows only). Description of the data, NOT a result.
Pw = T[T["window_pure"]]
print(Pw.groupby("label")[NEW_FEATURES].median().reindex(LABELS).round(3).to_string())
print("\nSame, by variant (medians). The absolute level (peer_median_level) and peer_level_rel should move DOWN for cooling variants:")
print(Pw.groupby("variant")[["agree_sign", "dev_mean_adj", "peer_level_rel", "dev_mean", "peer_median_level"]].median().reindex(VARIANT_ORDER).round(3).to_string())

             agree_sign  dev_mean_adj  peer_level_rel
label                                                
NORMAL            0.000         0.046           0.318
DRIFT             0.000         0.894           0.251
EVENT             0.089         0.025          -0.316
EVENT_DRIFT       0.089         0.884          -0.316

Same, by variant (medians). The absolute level (peer_median_level) and peer_level_rel should move DOWN for cooling variants:
              agree_sign  dev_mean_adj  peer_level_rel  dev_mean  peer_median_level
variant                                                                            
NORMAL             0.000         0.046           0.268    -0.057             21.721
DRIFT_RAMP         0.000         0.598           0.255     0.292             21.694
DRIFT_STEP         0.000         1.940           0.255     1.769             21.694
EVENT_WARM         0.081         0.053           3.161    -0.143             25.229
EVENT_COOL         0.088         0.015        

## 8. Save outputs and confirm inputs were not modified

In [22]:
def _num(v):
    return None if v is None or (isinstance(v, float) and np.isnan(v)) else float(v)

# Everything a gateway would need to compute the 3 new features: sigma_j per mote, offset per target, baseline per target and hour
REF_JSON = {
    "fit_split": "train only (real cleaned data, physical units, C)",
    "sigma_j": {str(k): _num(v) for k, v in REF["sigma"].items()},
    "offset": {str(t): (_num(s.median()) if len(s) else None) for t, s in REF["dev_daily"].items()},
    "baseline_by_hour": {str(t): {str(h): (_num(tab[h].median()) if h in tab.columns else None) for h in range(24)}
                         for t, tab in REF["hour_table"].items()},
    "note": "Validation/test windows use these final values. Training windows used leave-own-day-out versions of offset/baseline (see offset_used, baseline_used in the table).",
}
with open(PROC / "reference_stats_v2.json", "w") as f:
    json.dump(REF_JSON, f, indent=2)
if V1_REF_FILE.exists():
    v1ref = json.load(open(V1_REF_FILE)); v2ref = json.loads(json.dumps(REF_JSON))
    def close(a, b, tol=1e-5):
        if isinstance(a, dict):
            return a.keys() == b.keys() and all(close(a[k], b[k], tol) for k in a)
        if a is None or b is None:
            return a is None and b is None
        return abs(a - b) <= tol
    check(all(close(v1ref[k], v2ref[k]) for k in ["sigma_j", "offset", "baseline_by_hour"]), "reference statistics (sigma_j, offsets, hourly baselines) equal v1 within 1e-5 (float32 vs float64 rounding)")

AF_FILE = PROC / "feature_dataset_AF_v2.parquet"
T.to_parquet(AF_FILE, index=False)
CONFIG_JSON = {
    "status": "features only; no classifier was trained or evaluated here",
    "A_features": A_FEATURES, "F_features": F_FEATURES, "new_features": NEW_FEATURES, "all_features": ALL_FEATURES,
    "F_definition": "A minus [t_mean, t_min, t_max, peer_median_level] plus [agree_sign, dev_mean_adj, peer_level_rel]",
    "renamed": {"peer_agreement": "level_closeness (definition unchanged)"},
    "cfg": CFG,
    "primary_rows": "window_group == 'pure'; transition windows are kept and reported separately",
    "reference_statistics_file": "reference_stats_v2.json",
    "variants": VARIANT_ORDER, "v1_variants": V1_VARIANTS,
    "input_sha256": HASH_BEFORE, "real_data_file": REAL_FILE.name, "real_data_sha256": sha256(REAL_FILE),
    "n_windows": int(len(T)), "splits": T["split"].value_counts().to_dict(),
}
with open(PROC / "feature_config_AF_v2.json", "w") as f:
    json.dump(CONFIG_JSON, f, indent=2, default=str)

check({p.name: sha256(p) for p in READ_ONLY} == HASH_BEFORE, "NB3 v2 inputs and v1 comparison files unchanged by this notebook")
print("saved:", AF_FILE.name, f"({len(T)} rows)", "| feature_config_AF_v2.json | reference_stats_v2.json")

PASS  reference statistics (sigma_j, offsets, hourly baselines) equal v1 within 1e-5 (float32 vs float64 rounding)
PASS  NB3 v2 inputs and v1 comparison files unchanged by this notebook
saved: feature_dataset_AF_v2.parquet (11826 rows) | feature_config_AF_v2.json | reference_stats_v2.json


## 9. Summary

In [24]:
print("Windows        :", len(T), dict(T["split"].value_counts()))
print("Classes        :", T["label"].value_counts().reindex(LABELS).to_dict())
print("Variants       :", T["variant"].value_counts().reindex(VARIANT_ORDER).to_dict())
print("Pure/transition:", T["window_group"].value_counts().to_dict())
print("Set A          :", len(A_FEATURES), "features | Set F:", len(F_FEATURES), "features")
print("Reference stats: train-only, verified (garbage test, independent recomputation); saved in reference_stats_v2.json (equal to v1 within float32 rounding)")
print("Open limits    : 4 test episodes; voltage shift in the test period is not handled here; F is a candidate, not a proven improvement.")
print("Next           : NB5 v2: standard split on the 4 v1 variants (baseline), then cross-direction (train warming -> test cooling) and cross-drift-type (ramp -> step) tests.")

Windows        : 11826 {'train': 6570, 'validation': 2628, 'test': 2628}
Classes        : {'NORMAL': 1458, 'DRIFT': 2592, 'EVENT': 2592, 'EVENT_DRIFT': 5184}
Variants       : {'NORMAL': 1314, 'DRIFT_RAMP': 1314, 'DRIFT_STEP': 1314, 'EVENT_WARM': 1314, 'EVENT_COOL': 1314, 'ED_WARM_RAMP': 1314, 'ED_WARM_STEP': 1314, 'ED_COOL_RAMP': 1314, 'ED_COOL_STEP': 1314}
Pure/transition: {'pure': 11106, 'transition': 720}
Set A          : 16 features | Set F: 15 features
Reference stats: train-only, verified (garbage test, independent recomputation); saved in reference_stats_v2.json (equal to v1 within float32 rounding)
Open limits    : 4 test episodes; voltage shift in the test period is not handled here; F is a candidate, not a proven improvement.
Next           : NB5 v2: standard split on the 4 v1 variants (baseline), then cross-direction (train warming -> test cooling) and cross-drift-type (ramp -> step) tests.
